# ЛР2. Исследование моделей для предсказания цены автомобиля

В этом ноутбуке собраны основные исследовательские шаги по ЛР2. Чтобы не дублировать код, общие функции и строительные блоки пайплайнов вынесены в `research/common.py`, а ноутбук использует их через импорт.

Ноутбук охватывает:
- baseline-модель;
- генерацию новых признаков средствами `sklearn`;
- отбор признаков через `mlxtend.SequentialFeatureSelector`;
- подбор гиперпараметров для лучшей модели;
- выбор финальной Production-модели.

## Импорты и подготовка окружения

In [ ]:
from pathlib import Path
import sys

import optuna
import pandas as pd
from mlxtend.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from research.common import (
    CATEGORICAL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    FeatureIndexSelector,
    build_featured_preprocessor,
    build_featured_pipeline,
    build_baseline_pipeline,
    load_clean_dataset,
    load_selected_feature_indices,
    regression_metrics,
    split_dataset,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

## Загрузка данных

In [ ]:
dataset = load_clean_dataset()
split_data = split_dataset(dataset)

dataset.shape, split_data.X_train.shape, split_data.X_test.shape

In [ ]:
display(dataset.head())
print("Числовые признаки:", NUMERIC_FEATURES)
print("Категориальные признаки:", CATEGORICAL_FEATURES)
print("Целевая переменная:", TARGET_COLUMN)

## Baseline-модель

Baseline собирается как `StandardScaler + OrdinalEncoder + RandomForestRegressor`.

In [ ]:
baseline_pipeline = build_baseline_pipeline()
baseline_pipeline.fit(split_data.X_train, split_data.y_train)

baseline_predictions = baseline_pipeline.predict(split_data.X_test)
baseline_metrics = regression_metrics(split_data.y_test, baseline_predictions)
pd.DataFrame([baseline_metrics], index=["baseline"])

## Генерация дополнительных признаков средствами sklearn

На этом шаге используются:
- базовое шкалирование числовых признаков;
- `PolynomialFeatures(degree=2)`;
- `KBinsDiscretizer`;
- `OrdinalEncoder` для категориальных признаков.

In [ ]:
X_train_fe_sklearn = split_data.X_train.copy()
featured_preprocessor = build_featured_preprocessor()

X_train_fe_array = featured_preprocessor.fit_transform(X_train_fe_sklearn)
featured_feature_names = list(featured_preprocessor.get_feature_names_out())
X_train_fe_sklearn = pd.DataFrame(
    X_train_fe_array,
    columns=featured_feature_names,
    index=split_data.X_train.index,
)

X_train_fe_sklearn.head()

In [ ]:
print("Количество признаков после преобразований:", X_train_fe_sklearn.shape[1])
featured_feature_names

## Модель с дополнительными признаками

In [ ]:
featured_pipeline = build_featured_pipeline()
featured_pipeline.fit(split_data.X_train, split_data.y_train)

featured_predictions = featured_pipeline.predict(split_data.X_test)
featured_metrics = regression_metrics(split_data.y_test, featured_predictions)
pd.DataFrame([featured_metrics], index=["featured"])


## Forward-отбор признаков через mlxtend

Отбор выполняется на расширенном датафрейме `X_train_fe_sklearn`. В качестве доли отбираемых признаков используется 40% от общего числа признаков после sklearn-преобразований.

In [ ]:
total_features = X_train_fe_sklearn.shape[1]
selected_feature_count = max(5, round(total_features * 0.4))

sfs_estimator = RandomForestRegressor(
    random_state=42,
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=2,
    n_jobs=1,
)

sfs = SequentialFeatureSelector(
    sfs_estimator,
    k_features=selected_feature_count,
    forward=True,
    floating=False,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=1,
)
sfs.fit(X_train_fe_sklearn, split_data.y_train)

selected_indices = list(sfs.k_feature_idx_)
selected_feature_names = [featured_feature_names[index] for index in selected_indices]

print("Количество исходных преобразованных признаков:", total_features)
print("Количество отобранных признаков:", selected_feature_count)
print("Индексы отобранных признаков:", selected_indices)
selected_feature_names

## Модель с отбором признаков

In [ ]:
selected_pipeline = Pipeline(
    steps=[
        ("transform", build_featured_preprocessor()),
        ("select", FeatureIndexSelector(selected_indices)),
        (
            "regression",
            RandomForestRegressor(
                random_state=42,
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=2,
                n_jobs=1,
            ),
        ),
    ]
)

selected_pipeline.fit(split_data.X_train, split_data.y_train)
selected_predictions = selected_pipeline.predict(split_data.X_test)
selected_metrics = regression_metrics(split_data.y_test, selected_predictions)
pd.DataFrame([selected_metrics], index=["featured_sfs"])


## Подбор гиперпараметров лучшей модели

Для задачи регрессии целевая метрика здесь `MAE`, поэтому её необходимо **минимизировать**.

In [ ]:
TRIALS_COUNT = 10

def build_tunable_pipeline(selected_indices, n_estimators, max_depth, max_features):
    return Pipeline(
        steps=[
            ("transform", build_featured_preprocessor()),
            ("select", FeatureIndexSelector(selected_indices)),
            (
                "regression",
                RandomForestRegressor(
                    random_state=42,
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    max_features=max_features,
                    min_samples_leaf=2,
                    n_jobs=1,
                ),
            ),
        ]
    )

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 4, 20)
    max_features = trial.suggest_float("max_features", 0.1, 1.0)

    pipeline = build_tunable_pipeline(
        selected_indices=selected_indices,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
    )
    cv_scores = cross_val_score(
        pipeline,
        split_data.X_train,
        split_data.y_train,
        scoring="neg_mean_absolute_error",
        cv=3,
        n_jobs=1,
    )
    mae = -cv_scores.mean()
    return mae

study = optuna.create_study(
    study_name="rf_featured_sfs_mae_tuning",
    direction="minimize",
)
study.optimize(objective, n_trials=TRIALS_COUNT, show_progress_bar=False)

In [ ]:
best_params = study.best_params
best_params

In [ ]:
tuned_pipeline = build_tunable_pipeline(
    selected_indices=selected_indices,
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    max_features=best_params["max_features"],
)
tuned_pipeline.fit(split_data.X_train, split_data.y_train)

tuned_predictions = tuned_pipeline.predict(split_data.X_test)
tuned_metrics = regression_metrics(split_data.y_test, tuned_predictions)
pd.DataFrame([tuned_metrics], index=["tuned_sfs"])


## Сравнение моделей

In [ ]:
comparison = pd.DataFrame(
    [baseline_metrics, featured_metrics, selected_metrics, tuned_metrics],
    index=["baseline", "featured", "featured_sfs", "tuned_sfs"],
)
comparison.sort_values("mae")

## Выбор финальной модели

Если ориентироваться именно на тестовую метрику `MAE`, лучшей остаётся модель `featured_sfs`, а не tuned-версия. Поэтому для Production следует брать конфигурацию с forward-отбором признаков и параметрами:
- `n_estimators = 200`
- `max_depth = 12`
- `min_samples_leaf = 2`

Ниже подготовка пайплайна для финального обучения на всей очищенной выборке без повторного измерения метрик.

In [ ]:
production_selected_indices = load_selected_feature_indices()
X_full = dataset.drop(columns=[TARGET_COLUMN]).copy()
y_full = dataset[TARGET_COLUMN].copy()

production_pipeline = Pipeline(
    steps=[
        ("transform", build_featured_preprocessor()),
        ("select", FeatureIndexSelector(production_selected_indices)),
        (
            "regression",
            RandomForestRegressor(
                random_state=42,
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=2,
                n_jobs=1,
            ),
        ),
    ]
)

production_pipeline.fit(X_full, y_full)

## Артефакты на диске

Этот раздел помогает быстро соотнести ноутбук с сохранёнными артефактами и standalone-скриптами.

In [ ]:
research_dir = ROOT / "research"
sorted(path.name for path in research_dir.iterdir())